2D Heat Diffusion on the GPU: benchmark on a Colab T4. Choose Runtime > Change runtime type > T4 GPU, then run all cells.

In [ ]:
!nvidia-smi
from numba import cuda
assert cuda.is_available(), 'Select a T4 GPU runtime first'

Get the code from GitHub.

In [ ]:
REPO = 'https://github.com/Manishrepo-bi/2D-Heat-Diffusion-on-GPU.git'
BRANCH = 'main'

import os
if os.path.isdir('/content/heat/.git'):
    !git -C /content/heat fetch origin
    !git -C /content/heat checkout {BRANCH}
    !git -C /content/heat pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO} /content/heat
%cd /content/heat
!git log --oneline -3
!pip install -q -r requirements-cuda.txt

Known-answer checks and tests on the GPU.

In [ ]:
!python verify_all.py --backend cuda_global
!python verify_all.py --backend cuda_shared
!python -m pytest -q

Benchmark: CPU, global memory and shared memory.

In [ ]:
!python benchmark.py --sizes 256 512 1024 2048 --out results/benchmark_t4.csv

Tuning: block shape, cells per thread and float32.

In [ ]:
!python benchmark.py --sizes 1024 --blocks 16x16 32x8 32x16 --coarsen 1 2 4 --dtype float64 float32 --out results/tuning_t4.csv

Comparison of the two CSV files written by benchmark.py.

In [ ]:
!python compare_results.py